# Bayesian Linear Regression with Missing Responses (kidiq example)Goal: compare (a) full-data regression, (b) complete-case regression, and (c) Bayesian regression with missing responses using Gibbs sampling and imputation. All outputs and annotations are in English.

In [ ]:
# Imports and basic setupimport osimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy import statsimport statsmodels.api as smnp.random.seed(123)sns.set_style("whitegrid")plt.rcParams["figure.dpi"] = 120

## Load the kidiq data and perform basic inspectionThe dataset is expected to live in the same directory as this notebook. We keep only the response and chosen covariates, dropping rows with missing covariates so that only the response is incomplete later.

In [ ]:
# Load kidiq data and inspect the key columnsdata_path = "kidiq.dta"if not os.path.exists(data_path):    raise FileNotFoundError("The file 'kidiq.dta' was not found in the current directory.")df = pd.read_stata(data_path)# Choose covariates; feel free to adjust if you want to explore otherscovariate_cols = ["mom_hs", "mom_iq", "mom_work", "mom_age"]# Keep only the response and selected covariates; drop rows with missing covariatesselected_cols = ["kid_score"] + covariate_colsdf_clean = df[selected_cols].dropna(subset=covariate_cols).reset_index(drop=True)print(f"Rows after dropping missing covariates: {len(df_clean)}")display(df_clean.head())print("Summary of retained variables:")display(df_clean.describe(include="all"))

## Full-data regression (benchmark)Before introducing missing responses, fit an OLS model on the full data to obtain a benchmark for the coefficients and intervals.

In [ ]:
# Build full-data arrays and fit OLS as the ground-truth benchmarky_full = df_clean["kid_score"].to_numpy(dtype=float)X_full = sm.add_constant(df_clean[covariate_cols])ols_full = sm.OLS(y_full, X_full).fit()full_coef = ols_full.paramsfull_ci = ols_full.conf_int(alpha=0.05)full_summary = pd.DataFrame(    {        "coef": full_coef,        "ci_low": full_ci[0],        "ci_high": full_ci[1],    })print("Full-data OLS coefficients and 95% CIs (benchmark):")display(full_summary)

## Introduce missing responses under MCARRandomly remove a proportion of the responses (e.g., 20%) to simulate missing-at-random behavior while keeping covariates intact.

In [ ]:
# Create artificial missingness in the response vectormissing_rate = 0.20rng = np.random.default_rng(123)df_miss = df_clean.copy()n_obs = len(df_miss)missing_count = int(np.floor(n_obs * missing_rate))missing_indices = rng.choice(df_miss.index, size=missing_count, replace=False)df_miss.loc[missing_indices, "kid_score"] = np.nanobserved_count = n_obs - missing_countprint(f"Total observations: {n_obs}")print(f"Observed responses: {observed_count}")print(f"Missing responses: {missing_count}")df_miss.head()

## Construct design matrix and masks for observed/missing responsesSet up the design matrix with an intercept, create masks for observed and missing responses, and initialize missing responses with a sensible starting value.

In [ ]:
# Prepare design matrix and response with initial imputationsX = sm.add_constant(df_miss[covariate_cols]).to_numpy(dtype=float)y_with_nan = df_miss["kid_score"].to_numpy(dtype=float)obs_mask = ~np.isnan(y_with_nan)mis_mask = np.isnan(y_with_nan)# Initialize missing responses with draws around the observed meanobs_mean = np.nanmean(y_with_nan)obs_std = np.nanstd(y_with_nan)init_std = obs_std if obs_std > 0 else 1.0y_init = y_with_nan.copy()y_init[mis_mask] = rng.normal(loc=obs_mean, scale=init_std, size=mis_mask.sum())print(f"Design matrix shape: {X.shape}")print(f"Observed y count: {obs_mask.sum()}")print(f"Missing y count: {mis_mask.sum()}")

## Prior hyperparameters for Bayesian regressionWe use weakly informative priors for regression coefficients and variance: \(eta \sim N(0, cI)\) with a large scale \(c\), and \(\sigma^2 \sim 	ext{Inverse-Gamma}(a_0, b_0)\) with small shape/scale.

In [ ]:
# Specify weakly informative priorsn, p = X.shapebeta0 = np.zeros(p)V0 = np.eye(p) * 1e6V0_inv = np.linalg.inv(V0)a0 = 0.01b0 = 0.01

## Gibbs sampler implementationThe sampler cycles through updating the regression coefficients, variance, and missing responses. It returns posterior draws for \(eta\) and \(\sigma^2\).

In [ ]:
# Gibbs sampler for Bayesian linear regression with missing responsesdef gibbs_missing_y(y_with_nan, X, beta0, V0_inv, a0, b0, iterations=6000, burn_in=2000, rng_seed=123):    '''Run a Gibbs sampler for a Gaussian linear model with missing responses.    Parameters    ----------    y_with_nan : array-like        Response vector with NaNs indicating missing values.    X : array-like        Design matrix including the intercept column.    beta0 : array-like        Prior mean for beta.    V0_inv : array-like        Inverse prior covariance for beta.    a0 : float        Prior shape parameter for sigma^2 (Inverse-Gamma).    b0 : float        Prior scale parameter for sigma^2 (Inverse-Gamma).    iterations : int        Total number of Gibbs iterations.    burn_in : int        Number of initial samples to discard when summarizing.    rng_seed : int        Seed for reproducibility.    Returns    -------    beta_samples : ndarray        Samples of shape (iterations, p) for beta.    sigma2_samples : ndarray        Samples of shape (iterations,) for sigma^2.    final_y : ndarray        Imputed response vector from the last iteration.    '''    rng_local = np.random.default_rng(rng_seed)    y_current = np.array(y_with_nan, dtype=float)    obs_mask = ~np.isnan(y_current)    mis_mask = np.isnan(y_current)    # Initialize missing responses around the observed mean    obs_mean = np.nanmean(y_current)    obs_std = np.nanstd(y_current)    init_std = obs_std if obs_std > 0 else 1.0    y_current[mis_mask] = rng_local.normal(loc=obs_mean, scale=init_std, size=mis_mask.sum())    n, p = X.shape    beta_samples = np.zeros((iterations, p))    sigma2_samples = np.zeros(iterations)    # Simple OLS initialization using observed entries only    X_obs = X[obs_mask]    y_obs = y_current[obs_mask]    beta_current = np.linalg.lstsq(X_obs, y_obs, rcond=None)[0]    residuals = y_obs - X_obs @ beta_current    sigma2_current = np.var(residuals) if residuals.size > 1 else 1.0    for s in range(iterations):        # Update beta | sigma^2, y        Vn_inv = V0_inv + (1.0 / sigma2_current) * (X.T @ X)        Vn = np.linalg.inv(Vn_inv)        betan = Vn @ (V0_inv @ beta0 + (1.0 / sigma2_current) * (X.T @ y_current))        beta_current = rng_local.multivariate_normal(mean=betan, cov=Vn)        # Update sigma^2 | beta, y        resid = y_current - X @ beta_current        an = a0 + n / 2.0        bn = b0 + 0.5 * np.dot(resid, resid)        sigma2_current = stats.invgamma(a=an, scale=bn).rvs(random_state=rng_local)        # Update missing y | beta, sigma^2        if mis_mask.any():            mu_mis = X[mis_mask] @ beta_current            y_current[mis_mask] = rng_local.normal(loc=mu_mis, scale=np.sqrt(sigma2_current))        beta_samples[s] = beta_current        sigma2_samples[s] = sigma2_current    return beta_samples, sigma2_samples, y_current

## Run the Gibbs sampler and summarize the posteriorUse a moderate number of iterations for a smooth estimate, and summarize posterior means and 95% credible intervals after burn-in.

In [ ]:
# Execute the Gibbs samplergibbs_iterations = 6000burn_in = 2000beta_samples, sigma2_samples, y_imputed = gibbs_missing_y(    y_with_nan=y_with_nan,    X=X,    beta0=beta0,    V0_inv=V0_inv,    a0=a0,    b0=b0,    iterations=gibbs_iterations,    burn_in=burn_in,    rng_seed=321,)beta_posterior = beta_samples[burn_in:]sigma2_posterior = sigma2_samples[burn_in:]beta_mean = beta_posterior.mean(axis=0)beta_ci_low = np.quantile(beta_posterior, 0.025, axis=0)beta_ci_high = np.quantile(beta_posterior, 0.975, axis=0)sigma2_mean = sigma2_posterior.mean()sigma2_ci_low, sigma2_ci_high = np.quantile(sigma2_posterior, [0.025, 0.975])beta_bayes_summary = pd.DataFrame(    {        "coef": beta_mean,        "ci_low": beta_ci_low,        "ci_high": beta_ci_high,    },    index=["const"] + covariate_cols,)print("Posterior means and 95% credible intervals (Bayesian with missing y):")display(beta_bayes_summary)print(f"Sigma^2 posterior mean: {sigma2_mean:.3f} (95% CI: [{sigma2_ci_low:.3f}, {sigma2_ci_high:.3f}])")

## Complete-case regression for comparisonFit an OLS model using only rows where the response is observed after introducing missingness.

In [ ]:
# Complete-case OLS fitdf_complete = df_miss.dropna(subset=["kid_score"])y_cc = df_complete["kid_score"].to_numpy(dtype=float)X_cc = sm.add_constant(df_complete[covariate_cols])ols_cc = sm.OLS(y_cc, X_cc).fit()cc_coef = ols_cc.paramscc_ci = ols_cc.conf_int(alpha=0.05)cc_summary = pd.DataFrame(    {        "coef": cc_coef,        "ci_low": cc_ci[0],        "ci_high": cc_ci[1],    })print("Complete-case OLS coefficients and 95% CIs:")display(cc_summary)

## Side-by-side comparison of coefficient estimatesCompare the full-data benchmark, complete-case OLS, and Bayesian regression with missing-response imputation.

In [ ]:
# Assemble comparison table for coefficientscomparison = pd.DataFrame(index=["const"] + covariate_cols)comparison["beta_full"] = full_coefcomparison["beta_full_ci_low"] = full_ci[0]comparison["beta_full_ci_high"] = full_ci[1]comparison["beta_complete_case"] = cc_coefcomparison["beta_cc_ci_low"] = cc_ci[0]comparison["beta_cc_ci_high"] = cc_ci[1]comparison["beta_bayes_missing"] = beta_meancomparison["beta_bayes_ci_low"] = beta_ci_lowcomparison["beta_bayes_ci_high"] = beta_ci_highprint("Coefficient comparison across methods:")display(comparison)print("Sigma^2 comparison:")sigma2_comparison = pd.DataFrame(    {        "estimate": [ols_full.scale, ols_cc.scale, sigma2_mean],        "lower_95": [np.nan, np.nan, sigma2_ci_low],        "upper_95": [np.nan, np.nan, sigma2_ci_high],    },    index=["Full-data OLS", "Complete-case OLS", "Bayesian (missing y)"],)display(sigma2_comparison)

## Posterior diagnostics and saved plotsTrace plots and posterior densities for the regression coefficients and variance. PNG files are saved in the current directory for sharing.

In [ ]:
# Trace plots for beta coefficients and sigma^2trace_rows = len(beta_mean) + 1fig, axes = plt.subplots(trace_rows, 1, figsize=(8, 2 * trace_rows), sharex=True)coef_names = ["const"] + covariate_colsfor idx, name in enumerate(coef_names):    axes[idx].plot(beta_posterior[:, idx], color="steelblue", linewidth=0.7)    axes[idx].set_ylabel(name)axes[-1].plot(sigma2_posterior, color="darkred", linewidth=0.7)axes[-1].set_ylabel("sigma^2")axes[-1].set_xlabel("Iteration")fig.suptitle("Trace plots for Gibbs sampler", fontsize=12)plt.tight_layout()trace_path = "beta_traces.png"plt.savefig(trace_path, bbox_inches="tight")plt.close(fig)# Posterior density plotsfig, axes = plt.subplots(1, 2, figsize=(10, 4))for idx, name in enumerate(["const"] + covariate_cols[:1]):    sns.kdeplot(beta_posterior[:, idx], ax=axes[0], label=name)axes[0].set_title("Posterior densities for selected betas")axes[0].legend()sns.kdeplot(sigma2_posterior, ax=axes[1], color="darkred")axes[1].set_title("Posterior density for sigma^2")fig.suptitle("Posterior densities", fontsize=12)plt.tight_layout()density_path = "posterior_densities.png"plt.savefig(density_path, bbox_inches="tight")plt.close(fig)print(f"Trace plots saved to: {trace_path}")print(f"Posterior density plots saved to: {density_path}")